Name : NAVEEN GUPTA

Class : CSE A, 4th YEAR

Assignment : Anime Recommendation System Using TF-IDF and Cosine Similarity

DATASET LINK :

https://drive.google.com/drive/folders/1xAbz6RnqMINJ1BiFuCW88SYZhqd-xwff?usp=drive_link

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/agentic_winnovation/Anime_data.csv")

In [ ]:
df.columns

Index(['Anime_id', 'Title', 'Genre', 'Synopsis', 'Type', 'Producer', 'Studio',
       'Rating', 'ScoredBy', 'Popularity', 'Members', 'Episodes', 'Source',
       'Aired', 'Link'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17002 entries, 0 to 17001
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Anime_id    17002 non-null  int64  
 1   Title       17002 non-null  object 
 2   Genre       14990 non-null  object 
 3   Synopsis    15583 non-null  object 
 4   Type        16368 non-null  object 
 5   Producer    7635 non-null   object 
 6   Studio      7919 non-null   object 
 7   Rating      14425 non-null  float64
 8   ScoredBy    13227 non-null  float64
 9   Popularity  16368 non-null  float64
 10  Members     17002 non-null  float64
 11  Episodes    14085 non-null  float64
 12  Source      15075 non-null  object 
 13  Aired       16368 non-null  object 
 14  Link        16368 non-null  object 
dtypes: float64(5), int64(1), object(9)
memory usage: 1.9+ MB


In [ ]:
df.shape

(17002, 15)

In [ ]:
df.isnull().sum()

,0
Anime_id,0
Title,0
Genre,2012
Synopsis,1419
Type,634
Producer,9367
Studio,9083
Rating,2577
ScoredBy,3775
Popularity,634


In [ ]:
text_columns = [
    'Title',
    'Genre',
    'Synopsis',
    'Type',
    'Producer',
    'Studio',
    'Source'
]


deal null


In [ ]:

for col in text_columns:
    df[col] = df[col].fillna('')

In [ ]:
df['combined_features'] = (
    df['Genre'] + ' ' +
    df['Synopsis'] + ' ' +
    df['Type'] + ' ' +
    df['Producer'] + ' ' +
    df['Studio'] + ' ' +
    df['Source']
)

In [ ]:
list_columns = ['Genre', 'Producer', 'Studio']

for col in list_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[\[\]']", "", regex=True)
        .str.replace(",", " ", regex=False)
    )

In [ ]:
df['combine'] = (
    df['Title'].fillna('') + ' ' +
    df['Synopsis'].fillna('') + ' ' +
    df['Type'].fillna('') + ' ' +
    df['Source'].fillna('')
)

In [ ]:
print(df['combine'].head())

0    Cowboy Bebop In the year 2071, humanity has co...
1    Cowboy Bebop: Tengoku no Tobira Another day, a...
2    Trigun Vash the Stampede is the man with a $$6...
3    Witch Hunter Robin Witches are individuals wit...
4    Bouken Ou Beet It is the dark century and the ...
Name: combine, dtype: object


In [ ]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=15000
)

tfidf_matrix = tfidf.fit_transform(df['combine'])

In [ ]:
print(tfidf_matrix.shape)

(17002, 15000)


In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [ ]:
print(cosine_sim.shape)

(17002, 17002)


In [ ]:
indices = pd.Series(df.index, index=df['Title']).drop_duplicates()

In [ ]:
def recommend(title, top_n=10):

    if title not in indices:
        return "Anime not found."

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    anime_indices = [i[0] for i in sim_scores]

    return df.loc[
        anime_indices,
        ['Title', 'Rating']
    ]

In [ ]:
recommend("Cowboy Bebop")

,Title,Rating
1,Cowboy Bebop: Tengoku no Tobira,8.41
6541,Cowboy Bebop: Ein no Natsuyasumi,6.29
3180,Cowboy Bebop: Yose Atsume Blues,7.51
12459,Chibikko Cowboy,2.40
3289,Ginga Senpuu Braiger,6.93
12488,Waga Na wa Cowboy,4.00
848,Mutant Turtles: Choujin Densetsu-hen,5.36
15087,Seihou Bukyou Outlaw Star,7.98
6465,Solar I.II.III,5.07
12426,Blade Runner: Black Out 2022,7.08


In [ ]:
recommend("Naruto")

,Title,Rating
1434,Naruto: Shippuuden,8.17
15170,Naruto: Shippuuden Movie 6 - Road to Ninja,7.84
11327,Boruto: Naruto Next Generations,7.03
5585,Naruto: Honoo no Chuunin Shiken! Naruto vs. Ko...,7.26
10142,Boruto: Naruto the Movie - Naruto ga Hokage ni...,7.64
16183,Naruto: Shippuuden Movie 4 - The Lost Tower,7.53
3240,"Naruto: Shippuuden - Shippuu! ""Konoha Gakuen"" Den",7.26
8671,Boruto: Naruto the Movie,7.87
5551,Naruto: Shippuuden Movie 5 - Blood Prison,7.60
1901,Naruto: Dai Katsugeki!! Yuki Hime Shinobu Houj...,6.95


In [ ]:
def recommend(title, top_n=10):

    if title not in indices:
        return "Anime not found."

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    anime_indices = [i[0] for i in sim_scores]
    scores = [round(i[1],3) for i in sim_scores]

    result = df.loc[
        anime_indices,
        ['Title','Rating']
    ].copy()

    result['Similarity'] = scores

    return result

In [ ]:
recommend("Cowboy Bebop")

,Title,Rating,Similarity
1,Cowboy Bebop: Tengoku no Tobira,8.41,0.340
6541,Cowboy Bebop: Ein no Natsuyasumi,6.29,0.312
3180,Cowboy Bebop: Yose Atsume Blues,7.51,0.271
12459,Chibikko Cowboy,2.40,0.227
3289,Ginga Senpuu Braiger,6.93,0.195
12488,Waga Na wa Cowboy,4.00,0.174
848,Mutant Turtles: Choujin Densetsu-hen,5.36,0.167
15087,Seihou Bukyou Outlaw Star,7.98,0.143
6465,Solar I.II.III,5.07,0.122
12426,Blade Runner: Black Out 2022,7.08,0.119
